In [1]:
!pip install eurostat -q

In [2]:
import eurostat
import pandas as pd

# Tablonun parametrelerini öğren
print("Parametreler:", eurostat.get_pars('sts_trtu_m'))

# Geo listesinde 4 ülkemiz var mı?
geos = eurostat.get_dic('sts_trtu_m', 'geo')
ours = [g for g in geos if g[0] in ['DE', 'ES', 'TR', 'UK', 'GB']]
print("\nBizim ülkeler:", ours)

Parametreler: ['freq', 'indic_bt', 'nace_r2', 's_adj', 'unit', 'geo']

Bizim ülkeler: [('DE', 'Germany'), ('ES', 'Spain'), ('UK', 'United Kingdom'), ('TR', 'Türkiye')]


In [3]:
print("indic_bt değerleri:")
for code, label in eurostat.get_dic('sts_trtu_m', 'indic_bt'):
    print(f"  {code:8s} → {label}")

print("\nunit değerleri:")
for code, label in eurostat.get_dic('sts_trtu_m', 'unit')[:10]:
    print(f"  {code:8s} → {label}")

print("\nnace_r2 değerleri (ilk 8):")
for code, label in eurostat.get_dic('sts_trtu_m', 'nace_r2')[:8]:
    print(f"  {code:8s} → {label}")

indic_bt değerleri:
  TOTAL    → Total
  PRD      → Production (volume)
  VOL_SLS  → Volume of sales
  NETTUR   → Net turnover
  NETTUR_DOM → Domestic net turnover
  NETTUR_NDOM → Non-domestic net turnover
  NETTUR_NDOM_EU → Non-domestic net turnover - euro area
  NETTUR_NDOM_NEU → Non-domestic net turnover - non-euro area
  EMP      → Persons employed
  HW       → Hours worked by employees
  WAGE     → Wages and salaries
  PRC_IMP  → Import prices
  PRC_IMP_EU → Import prices - euro area
  PRC_IMP_NEU → Import prices - non-euro area
  PRC_PRR_DOM → Domestic producer prices
  PRC_PRR_NDOM → Non-domestic producer prices
  PRC_PRR_NDOM_EU → Non-domestic producer prices - euro area
  PRC_PRR_NDOM_NEU → Non-domestic producer prices - non-euro area
  PRC_PRR  → Producer prices
  PRC_PRR_B2B → Producer prices - business-to business
  COST     → Costs
  FIN_SIT  → Financial situation of the business
  FIN_COS  → Cost (interest and other) of obtaining finance
  FIN_DEB  → Debt/turnover ratio o

In [4]:
my_filter = {
    'startPeriod': '2015-01',
    'endPeriod':   '2024-12',
    'geo':         ['DE', 'ES', 'TR', 'UK'],
    'freq':        'M',
    'indic_bt':    'VOL_SLS',   # Volume of sales (hacim endeksi)
    's_adj':       'SCA',        # seasonally + working-day adjusted
    'unit':        'I21',        # endeks, 2021=100
    'nace_r2':     'G47'         # toplam perakende ticaret
}

df_retail = eurostat.get_data_df('sts_trtu_m', filter_pars=my_filter)
print("Shape:", df_retail.shape)
print("Sütunlar:", df_retail.columns.tolist()[:10])
df_retail.head()

Shape: (3, 126)
Sütunlar: ['freq', 'indic_bt', 'nace_r2', 's_adj', 'unit', 'geo\\TIME_PERIOD', '2015-01', '2015-02', '2015-03', '2015-04']


,freq,indic_bt,nace_r2,s_adj,unit,geo\TIME_PERIOD,2015-01,2015-02,2015-03,2015-04,...,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,M,VOL_SLS,G47,SCA,I21,DE,84.5,84.5,84.3,84.9,...,96.3,96.5,96.6,95.2,97.0,98.2,99.7,99.2,99.1,99.1
1,M,VOL_SLS,G47,SCA,I21,ES,93.6,93.5,94.3,93.7,...,105.2,105.9,105.4,105.4,106.4,106.9,107.9,108.1,107.0,109.4
2,M,VOL_SLS,G47,SCA,I21,TR,76.1,76.2,77.3,77.4,...,156.4,153.6,152.6,155.6,158.1,160.5,164.1,164.7,167.4,168.4


In [5]:
# Wide formatından long formata
id_cols = ['freq', 'indic_bt', 'nace_r2', 's_adj', 'unit', 'geo\\TIME_PERIOD']
retail_long = df_retail.melt(
    id_vars=id_cols,
    var_name='time',
    value_name='retail_index'
)

# Geo sütunu adı tuhaf ('geo\\TIME_PERIOD'), düzelt
retail_long = retail_long.rename(columns={'geo\\TIME_PERIOD': 'geo'})

# Ülke kodlarını projendeki isimlere eşle
country_map = {'DE': 'Germany', 'ES': 'Spain', 'TR': 'Turkey'}
retail_long['country'] = retail_long['geo'].map(country_map)

# Tarih + temizlik
retail_long['year_month'] = pd.to_datetime(retail_long['time']).dt.to_period('M')
retail_long = retail_long[['country', 'year_month', 'retail_index']].dropna()
retail_long = retail_long.sort_values(['country', 'year_month']).reset_index(drop=True)

print("Final shape:", retail_long.shape)
print("Ülke başına:")
print(retail_long.groupby('country').size())
print("\nİlk 5 satır:")
print(retail_long.head())
print("\nSon 5 satır:")
print(retail_long.tail())

Final shape: (360, 3)
Ülke başına:
country
Germany    120
Spain      120
Turkey     120
dtype: int64

İlk 5 satır:
   country year_month  retail_index
0  Germany    2015-01          84.5
1  Germany    2015-02          84.5
2  Germany    2015-03          84.3
3  Germany    2015-04          84.9
4  Germany    2015-05          86.0

Son 5 satır:
    country year_month  retail_index
355  Turkey    2024-08         160.5
356  Turkey    2024-09         164.1
357  Turkey    2024-10         164.7
358  Turkey    2024-11         167.4
359  Turkey    2024-12         168.4


In [6]:
import os
os.makedirs('data', exist_ok=True)
retail_long.to_csv('data/retail_sales_monthly.csv', index=False)
print("✅ data/retail_sales_monthly.csv kaydedildi")

✅ data/retail_sales_monthly.csv kaydedildi


In [7]:
import requests
import time
import pandas as pd

# Ülke başına 5 büyük şehir + nüfus (binler)
cities = {
    "Germany": [
        ("Berlin",    52.52, 13.40, 3677),
        ("Hamburg",   53.55, 10.00, 1900),
        ("Munich",    48.13, 11.58, 1488),
        ("Cologne",   50.94,  6.96, 1086),
        ("Frankfurt", 50.11,  8.68,  763),
    ],
    "Spain": [
        ("Madrid",    40.42, -3.70, 3300),
        ("Barcelona", 41.39,  2.17, 1620),
        ("Valencia",  39.47, -0.38,  791),
        ("Sevilla",   37.39, -5.99,  684),
        ("Bilbao",    43.26, -2.93,  346),
    ],
    "Turkey": [
        ("Istanbul",  41.01, 28.98, 15462),
        ("Ankara",    39.93, 32.85,  5663),
        ("Izmir",     38.42, 27.14,  4394),
        ("Antalya",   36.90, 30.71,  2548),
        ("Trabzon",   41.00, 39.72,   811),
    ],
    "UK": [
        ("London",     51.51, -0.13, 8982),
        ("Manchester", 53.48, -2.24,  553),
        ("Birmingham", 52.49, -1.90, 1141),
        ("Leeds",      53.80, -1.55,  789),
        ("Edinburgh",  55.95, -3.19,  548),
    ],
}

def fetch_city_weather(lat, lon, start='2015-01-01', end='2024-12-31'):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": ",".join([
            "sunshine_duration", "daylight_duration",
            "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
            "precipitation_sum", "rain_sum", "precipitation_hours"
        ]),
        "timezone": "UTC"
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["daily"])
    df["time"] = pd.to_datetime(df["time"])
    return df

In [9]:
import requests
import time
import pandas as pd

def fetch_city_weather_with_retry(lat, lon, max_retries=4):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": '2015-01-01', "end_date": '2024-12-31',
        "daily": ",".join([
            "sunshine_duration", "daylight_duration",
            "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
            "precipitation_sum", "rain_sum", "precipitation_hours"
        ]),
        "timezone": "UTC"
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 429:
                wait = 60 * (attempt + 1)
                print(f"    ⚠️  429 rate limit. {wait}s bekleniyor...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            df = pd.DataFrame(r.json()["daily"])
            df["time"] = pd.to_datetime(df["time"])
            return df
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"    ⚠️  Hata: {e}. Tekrar deneniyor...")
            time.sleep(30)
    raise Exception("Max retries aşıldı")

# Mevcut all_records varsa, hangileri zaten çekildi?
if 'all_records' not in dir():
    all_records = []

fetched_cities = set()
for df in all_records:
    if 'city' in df.columns and len(df) > 0:
        fetched_cities.add(df['city'].iloc[0])
print(f"Zaten çekilenler: {fetched_cities}\n")

# Kalan şehirleri çek
for country, city_list in cities.items():
    for name, lat, lon, pop in city_list:
        if name in fetched_cities:
            print(f"  ✓ {country}/{name} (atlandı)")
            continue
        print(f"  → {country}/{name} çekiliyor...")
        df = fetch_city_weather_with_retry(lat, lon)
        df["country"] = country
        df["city"] = name
        df["population"] = pop
        all_records.append(df)
        time.sleep(2.5)  # her başarılı istekten sonra 2.5s bekle

weather_daily = pd.concat(all_records, ignore_index=True)
print(f"\n✅ Toplam günlük: {weather_daily.shape}")
print(f"Şehir sayısı: {weather_daily['city'].nunique()}")

Zaten çekilenler: {'Munich', 'Berlin', 'Cologne', 'Hamburg'}

  ✓ Germany/Berlin (atlandı)
  ✓ Germany/Hamburg (atlandı)
  ✓ Germany/Munich (atlandı)
  ✓ Germany/Cologne (atlandı)
  → Germany/Frankfurt çekiliyor...
  → Spain/Madrid çekiliyor...
  → Spain/Barcelona çekiliyor...
  → Spain/Valencia çekiliyor...
    ⚠️  429 rate limit. 60s bekleniyor...
  → Spain/Sevilla çekiliyor...
  → Spain/Bilbao çekiliyor...
  → Turkey/Istanbul çekiliyor...
    ⚠️  429 rate limit. 60s bekleniyor...
    ⚠️  Hata: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out.. Tekrar deneniyor...
  → Turkey/Ankara çekiliyor...
  → Turkey/Izmir çekiliyor...
  → Turkey/Antalya çekiliyor...
    ⚠️  429 rate limit. 60s bekleniyor...
  → Turkey/Trabzon çekiliyor...
  → UK/London çekiliyor...
  → UK/Manchester çekiliyor...
  → UK/Birmingham çekiliyor...
    ⚠️  429 rate limit. 60s bekleniyor...
  → UK/Leeds çekiliyor...
  → UK/Edinburgh çekiliyor...

✅ Toplam günlük: (73060, 12)
Şehir sayıs

In [10]:
weather_daily["year_month"] = weather_daily["time"].dt.to_period("M")

# Şehir × ay seviyesinde özetle
monthly_city = weather_daily.groupby(
    ["country", "city", "population", "year_month"], as_index=False
).agg({
    "sunshine_duration":   "sum",
    "daylight_duration":   "sum",
    "temperature_2m_mean": "mean",
    "temperature_2m_max":  "max",
    "temperature_2m_min":  "min",
    "precipitation_sum":   "sum",
    "rain_sum":            "sum",
    "precipitation_hours": "sum",
})

# Saniyeleri saate çevir
monthly_city["sunshine_hours"] = monthly_city["sunshine_duration"] / 3600
monthly_city["daylight_hours"] = monthly_city["daylight_duration"] / 3600
monthly_city = monthly_city.drop(columns=["sunshine_duration", "daylight_duration"])

# Nüfus ağırlıklı ülke seviyesi
def weighted_avg(g, col):
    return (g[col] * g["population"]).sum() / g["population"].sum()

cols_to_avg = ["sunshine_hours","daylight_hours",
               "temperature_2m_mean","temperature_2m_max","temperature_2m_min",
               "precipitation_sum","rain_sum","precipitation_hours"]

weather_country = (
    monthly_city.groupby(["country","year_month"])
    .apply(lambda g: pd.Series({c: weighted_avg(g, c) for c in cols_to_avg}))
    .reset_index()
)

print(f"✅ Ülke aylık: {weather_country.shape}")
print(weather_country.head())
weather_country.to_csv('data/weather_monthly.csv', index=False)
print("✅ data/weather_monthly.csv kaydedildi")

✅ Ülke aylık: (480, 10)
   country year_month  sunshine_hours  daylight_hours  temperature_2m_mean  \
0  Germany    2015-01       90.455698      260.506157             2.497282   
1  Germany    2015-02      170.676347      278.978642             1.210904   
2  Germany    2015-03      241.151376      368.355612             5.518422   
3  Germany    2015-04      324.719014      416.586308             8.589733   
4  Germany    2015-05      335.578942      484.918020            12.899274   

   temperature_2m_max  temperature_2m_min  precipitation_sum   rain_sum  \
0           11.886583           -4.188389          89.431714  73.215751   
1            9.801223           -7.359390          18.682140  12.722863   
2           15.600931           -2.387996          56.812194  53.475510   
3           20.832701           -2.643774          37.266491  36.500684   
4           25.144043            3.356114          66.306686  66.306686   

   precipitation_hours  
0           191.631254  
1     

/var/folders/4x/pgcdn1lj1z708hjlhy1sq_680000gn/T/ipykernel_7458/2298050192.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({c: weighted_avg(g, c) for c in cols_to_avg}))


In [15]:
unemp_filter = {
    'startPeriod': '2015-01', 'endPeriod': '2024-12',
    'geo': ['DE', 'ES', 'TR', 'UK'],
    'freq': 'M',
    'age': 'TOTAL',        # ← Y15-74 değil
    'sex': 'T',
    'unit': 'PC_ACT',
    's_adj': 'SA'
}

df_unemp = eurostat.get_data_df('une_rt_m', filter_pars=unemp_filter)
print("Unemployment shape:", df_unemp.shape)
print("Ülkeler:", df_unemp['geo\\TIME_PERIOD'].unique() if 'geo\\TIME_PERIOD' in df_unemp.columns else df_unemp.iloc[:,5].unique())
df_unemp.head()

Unemployment shape: (4, 126)
Ülkeler: ['DE' 'ES' 'TR' 'UK']


,freq,s_adj,age,unit,sex,geo\TIME_PERIOD,2015-01,2015-02,2015-03,2015-04,...,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,M,SA,TOTAL,PC_ACT,T,DE,4.5,4.5,4.5,4.4,...,3.3,3.3,3.4,3.4,3.4,3.4,3.4,3.4,3.4,3.5
1,M,SA,TOTAL,PC_ACT,T,ES,23.4,23.1,22.9,22.7,...,11.6,11.6,11.6,11.5,11.4,11.3,11.1,10.9,10.8,10.8
2,M,SA,TOTAL,PC_ACT,T,TR,10.5,10.7,10.5,10.5,...,8.8,8.6,8.4,9.1,9.0,8.5,8.5,8.7,8.4,8.6
3,M,SA,TOTAL,PC_ACT,T,UK,5.5,5.5,5.4,5.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Tek ülke (DE), tek ay, tüm filtreler açık
test_filter = {
    'startPeriod': '2024-01', 'endPeriod': '2024-12',
    'geo': 'DE',
    'freq': 'M',
}
df_test = eurostat.get_data_df('une_rt_m', filter_pars=test_filter)
print("Shape:", df_test.shape)
print("Sütunlar:", df_test.columns.tolist())
print("\nÖrnek satırlar (her age/sex/s_adj kombinasyonu):")
print(df_test[['age','sex','unit','s_adj']].drop_duplicates().head(20))

Shape: (54, 18)
Sütunlar: ['freq', 's_adj', 'age', 'unit', 'sex', 'geo\\TIME_PERIOD', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

Örnek satırlar (her age/sex/s_adj kombinasyonu):
       age sex     unit s_adj
0    TOTAL   F   PC_ACT   NSA
1    TOTAL   M   PC_ACT   NSA
2    TOTAL   T   PC_ACT   NSA
3    TOTAL   F  THS_PER   NSA
4    TOTAL   M  THS_PER   NSA
5    TOTAL   T  THS_PER   NSA
6   Y25-74   F   PC_ACT   NSA
7   Y25-74   M   PC_ACT   NSA
8   Y25-74   T   PC_ACT   NSA
9   Y25-74   F  THS_PER   NSA
10  Y25-74   M  THS_PER   NSA
11  Y25-74   T  THS_PER   NSA
12  Y_LT25   F   PC_ACT   NSA
13  Y_LT25   M   PC_ACT   NSA
14  Y_LT25   T   PC_ACT   NSA
15  Y_LT25   F  THS_PER   NSA
16  Y_LT25   M  THS_PER   NSA
17  Y_LT25   T  THS_PER   NSA
18   TOTAL   F   PC_ACT    SA
19   TOTAL   M   PC_ACT    SA


In [16]:
# wide_to_long fonksiyonu daha önce tanımlandıysa gerek yok, yoksa yeniden tanımla:
def wide_to_long(df, value_name):
    id_cols = [c for c in df.columns if not c[0].isdigit()]
    geo_col = [c for c in id_cols if 'geo' in c.lower()][0]
    df_long = df.melt(id_vars=id_cols, var_name='time', value_name=value_name)
    df_long = df_long.rename(columns={geo_col: 'geo'})
    country_map = {'DE':'Germany', 'ES':'Spain', 'TR':'Turkey', 'UK':'UK'}
    df_long['country'] = df_long['geo'].map(country_map)
    df_long['year_month'] = pd.to_datetime(df_long['time']).dt.to_period('M')
    return df_long[['country', 'year_month', value_name]].dropna(
        subset=['country']).sort_values(['country','year_month']).reset_index(drop=True)

unemp_long = wide_to_long(df_unemp, 'unemployment_rate')

print("Per country (NaN dahil):")
print(unemp_long.groupby('country').size())
print("\nNaN sayısı:")
print(unemp_long.groupby('country')['unemployment_rate'].apply(lambda x: x.isna().sum()))
print("\nİlk satırlar:")
print(unemp_long.head())

Per country (NaN dahil):
country
Germany    120
Spain      120
Turkey     120
UK         120
dtype: int64

NaN sayısı:
country
Germany     0
Spain       0
Turkey      0
UK         51
Name: unemployment_rate, dtype: int64

İlk satırlar:
   country year_month  unemployment_rate
0  Germany    2015-01                4.5
1  Germany    2015-02                4.5
2  Germany    2015-03                4.5
3  Germany    2015-04                4.4
4  Germany    2015-05                4.4


In [19]:
infl_filter = {
    'startPeriod': '2015-01', 'endPeriod': '2024-12',
    'geo': ['DE', 'ES', 'TR', 'UK'],
    'freq': 'M',
    'coicop': 'CP00',
    'unit': 'RCH_A'      # RCH_A1 değil!
}

df_infl = eurostat.get_data_df('prc_hicp_manr', filter_pars=infl_filter)
print("Inflation shape:", df_infl.shape)
df_infl.head()


Inflation shape: (4, 124)


,freq,unit,coicop,geo\TIME_PERIOD,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,...,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,M,RCH_A,CP00,DE,-0.5,-0.2,0.3,1.0,1.6,1.1,...,2.3,2.4,2.8,2.5,2.6,2.0,1.8,2.4,2.4,2.8
1,M,RCH_A,CP00,ES,-1.5,-1.2,-0.8,-0.7,-0.3,0.0,...,3.3,3.4,3.8,3.6,2.9,2.4,1.7,1.8,2.4,2.8
2,M,RCH_A,CP00,TR,7.2,7.6,7.6,7.9,8.3,7.6,...,68.6,69.8,75.5,71.6,61.8,52.0,49.5,48.7,47.1,44.4
3,M,RCH_A,CP00,UK,0.3,0.0,0.0,-0.1,0.1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
def wide_to_long(df, value_name):
    id_cols = [c for c in df.columns if not c[0].isdigit()]
    geo_col = [c for c in id_cols if 'geo' in c.lower()][0]
    df_long = df.melt(id_vars=id_cols, var_name='time', value_name=value_name)
    df_long = df_long.rename(columns={geo_col: 'geo'})
    country_map = {'DE':'Germany', 'ES':'Spain', 'TR':'Turkey', 'UK':'UK'}
    df_long['country'] = df_long['geo'].map(country_map)
    df_long['year_month'] = pd.to_datetime(df_long['time']).dt.to_period('M')
    return df_long[['country','year_month',value_name]].dropna(
        subset=['country']).sort_values(['country','year_month']).reset_index(drop=True)

infl_long = wide_to_long(df_infl, 'inflation_yoy')
print(infl_long.groupby('country').size())

print("\nNaN count:")
print(infl_long.groupby('country')['inflation_yoy'].apply(lambda x: x.isna().sum()))
infl_long.head()


country
Germany    120
Spain      120
Turkey     120
UK         120
dtype: int64

NaN count:
country
Germany     0
Spain       0
Turkey      0
UK         49
Name: inflation_yoy, dtype: int64


,country,year_month,inflation_yoy
0,Germany,2015-01,-0.5
1,Germany,2015-02,-0.2
2,Germany,2015-03,0.3
3,Germany,2015-04,1.0
4,Germany,2015-05,1.6


In [22]:
cci_raw = pd.read_csv('oecd_cci_monthly.csv.csv')
countries_map = {
    "United Kingdom": "UK",
    "Germany":        "Germany",
    "Spain":          "Spain",
    "Türkiye":        "Turkey"
}
cci = cci_raw[cci_raw['Reference area'].isin(countries_map.keys())][
    ['Reference area','TIME_PERIOD','OBS_VALUE']
].copy()
cci['country'] = cci['Reference area'].map(countries_map)
cci['year_month'] = pd.to_datetime(cci['TIME_PERIOD']).dt.to_period('M')
cci['CCI'] = pd.to_numeric(cci['OBS_VALUE'], errors='coerce')
cci_long = cci[['country','year_month','CCI']].dropna()
cci_long = cci_long[(cci_long['year_month'] >= '2015-01') & 
                    (cci_long['year_month'] <= '2024-12')]
print("CCI per country:")
print(cci_long.groupby('country').size())

CCI per country:
country
Germany    120
Spain      120
Turkey     120
UK         120
dtype: int64


In [24]:
# weather_country zaten elimizde (Adım 1b'den)
# retail_long, unemp_long, infl_long, cci_long da hazır

panel = (weather_country
    .merge(retail_long, on=['country','year_month'], how='left')
    .merge(unemp_long,  on=['country','year_month'], how='left')
    .merge(infl_long,   on=['country','year_month'], how='left')
    .merge(cci_long,    on=['country','year_month'], how='left'))

# COVID dummy: 2020-03 ile 2022-06 arası
panel['covid_dummy'] = panel['year_month'].apply(
    lambda ym: 1 if pd.Period('2020-03') <= ym <= pd.Period('2022-06') else 0
)

# Mevsim sütunları (sin/cos encoding) — month_sin, month_cos
panel['month_num'] = panel['year_month'].dt.month
import numpy as np
panel['month_sin'] = np.sin(2 * np.pi * panel['month_num'] / 12)
panel['month_cos'] = np.cos(2 * np.pi * panel['month_num'] / 12)

print(f"Panel shape: {panel.shape}")
print(f"Sütunlar: {panel.columns.tolist()}")
print(f"\nÜlke başına satır:")
print(panel.groupby('country').size())
print(f"\nNaN sayısı her sütunda:")
print(panel.isna().sum())


Panel shape: (480, 18)
Sütunlar: ['country', 'year_month', 'sunshine_hours', 'daylight_hours', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'retail_index', 'unemployment_rate', 'inflation_yoy', 'CCI', 'covid_dummy', 'month_num', 'month_sin', 'month_cos']

Ülke başına satır:
country
Germany    120
Spain      120
Turkey     120
UK         120
dtype: int64

NaN sayısı her sütunda:
country                  0
year_month               0
sunshine_hours           0
daylight_hours           0
temperature_2m_mean      0
temperature_2m_max       0
temperature_2m_min       0
precipitation_sum        0
rain_sum                 0
precipitation_hours      0
retail_index           120
unemployment_rate       51
inflation_yoy           49
CCI                      0
covid_dummy              0
month_num                0
month_sin                0
month_cos                0
dtype: int64


In [25]:
import os
os.makedirs('data', exist_ok=True)

# year_month Period tipinden string'e çevir (parquet uyumluluğu)
panel_to_save = panel.copy()
panel_to_save['year_month'] = panel_to_save['year_month'].astype(str)

panel_to_save.to_parquet('data/panel.parquet', index=False)
panel_to_save.to_csv('data/panel.csv', index=False)

print("✅ data/panel.parquet kaydedildi")
print("✅ data/panel.csv kaydedildi (yedek)")
print(f"\nFinal shape: {panel_to_save.shape}")
panel_to_save.head()


✅ data/panel.parquet kaydedildi
✅ data/panel.csv kaydedildi (yedek)

Final shape: (480, 18)


,country,year_month,sunshine_hours,daylight_hours,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,retail_index,unemployment_rate,inflation_yoy,CCI,covid_dummy,month_num,month_sin,month_cos
0,Germany,2015-01,90.455698,260.506157,2.497282,11.886583,-4.188389,89.431714,73.215751,191.631254,84.5,4.5,-0.5,99.64577,0,1,0.500000,8.660254e-01
1,Germany,2015-02,170.676347,278.978642,1.210904,9.801223,-7.359390,18.682140,12.722863,53.187121,84.5,4.5,-0.2,99.73933,0,2,0.866025,5.000000e-01
2,Germany,2015-03,241.151376,368.355612,5.518422,15.600931,-2.387996,56.812194,53.475510,124.640565,84.3,4.5,0.3,99.83885,0,3,1.000000,6.123234e-17
3,Germany,2015-04,324.719014,416.586308,8.589733,20.832701,-2.643774,37.266491,36.500684,91.627889,84.9,4.4,1.0,99.92391,0,4,0.866025,-5.000000e-01
4,Germany,2015-05,335.578942,484.918020,12.899274,25.144043,3.356114,66.306686,66.306686,109.135966,86.0,4.4,1.6,99.97682,0,5,0.500000,-8.660254e-01
